In [ ]:
# Cell 0: Load MBPP dataset and inspect format
from datasets import load_dataset

full = load_dataset("google-research-datasets/mbpp", "full")
print("Splits:", list(full.keys()))
print("Total examples:", len(full["train"]))
print("\nColumns:", full["train"].column_names)
print("\n--- Example (task_id=1) ---")
ex = full["train"][0]
for k, v in ex.items():
    print(f"  {k}: {repr(v)[:200]}")

In [ ]:
# Cell 1: Split into train/test by task_id (same logic as grpo_code.py)
all_data = full["train"]

train_ids = set(range(1, 601)) | set(range(701, 975))
test_ids = set(range(601, 701))

train_data = all_data.filter(lambda x: x["task_id"] in train_ids)
test_data = all_data.filter(lambda x: x["task_id"] in test_ids)

print(f"Train: {len(train_data)}, Test: {len(test_data)}")
print(f"\n--- Sample train examples ---")
for i in range(3):
    ex = train_data[i]
    print(f"\n[task_id={ex['task_id']}] {ex['text'][:120]}...")
    print(f"  tests: {ex['test_list']}")
    print(f"  code:  {ex['code'][:100]}...")

In [ ]:
# Cell 2: Test code extraction logic
import re

def extract_code(text: str) -> str:
    """Extract Python code from completion. Tries ```python blocks first, then raw."""
    blocks = re.findall(r"```(?:python)?\s*\n(.*?)```", text, re.DOTALL)
    if blocks:
        return blocks[-1].strip()
    for marker in ["```", "def ", "import "]:
        idx = text.find(marker)
        if idx != -1:
            candidate = text[idx:].strip()
            if candidate.startswith("```"):
                candidate = candidate[3:].strip()
                if candidate.startswith("python"):
                    candidate = candidate[6:].strip()
            return candidate
    think_end = text.find("</think>")
    if think_end != -1:
        return text[think_end + 8:].strip()
    return text.strip()

# Test cases
test_completions = [
    # Case 1: proper fenced block
    """<think>
I need to write a function to find the minimum cost path.
</think>

```python
def min_cost(cost, m, n):
    tc = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        for j in range(n + 1):
            if i == 0 and j == 0:
                tc[i][j] = cost[i][j]
            elif i == 0:
                tc[i][j] = tc[i][j-1] + cost[i][j]
            elif j == 0:
                tc[i][j] = tc[i-1][j] + cost[i][j]
            else:
                tc[i][j] = min(tc[i-1][j-1], tc[i-1][j], tc[i][j-1]) + cost[i][j]
    return tc[m][n]
```""",
    # Case 2: no fenced block, raw def
    """def add(a, b):
    return a + b""",
    # Case 3: just gibberish (should return as-is)
    "I don't know how to code this sorry",
]

for i, comp in enumerate(test_completions):
    code = extract_code(comp)
    print(f"--- Case {i+1} ---")
    print(code[:200])
    print()

In [ ]:
# Cell 3: Test execution-based reward
import subprocess
import sys
import tempfile
import os

def execute_code_with_tests(code: str, tests: list[str], timeout: int = 10) -> bool:
    """Execute code + test assertions in a subprocess with timeout."""
    test_block = "\n".join(tests)
    full_script = f"{code}\n\n{test_block}\n"
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(full_script)
            tmp_path = f.name
        result = subprocess.run(
            [sys.executable, tmp_path],
            capture_output=True, text=True, timeout=timeout,
        )
        return result.returncode == 0
    except (subprocess.TimeoutExpired, Exception):
        return False
    finally:
        try:
            os.unlink(tmp_path)
        except OSError:
            pass

# Test with ground-truth code from dataset
print("=== Execution tests with ground-truth code ===\n")
for i in range(5):
    ex = train_data[i]
    passed = execute_code_with_tests(ex["code"], ex["test_list"])
    status = "PASS" if passed else "FAIL"
    print(f"[task_id={ex['task_id']}] {status} | {ex['text'][:80]}...")
    if not passed:
        # Debug: show what went wrong
        test_block = "\n".join(ex["test_list"])
        full_script = f"{ex['code']}\n\n{test_block}\n"
        r = subprocess.run([sys.executable, "-c", full_script], capture_output=True, text=True, timeout=10)
        print(f"  stderr: {r.stderr[:200]}")

# Test with bad code
print("\n=== Execution tests with bad code ===")
bad_code = "def add(a, b):\n    return a - b  # wrong!"
bad_tests = ["assert add(1, 2) == 3"]
print(f"Bad code pass: {execute_code_with_tests(bad_code, bad_tests)}")  # should be False

# Test with timeout (infinite loop)
loop_code = "while True: pass"
print(f"Infinite loop pass: {execute_code_with_tests(loop_code, ['assert True'], timeout=2)}")  # should be False

In [ ]:
# Cell 4: Test all three reward functions together
def correctness_reward(completions, test_list, **kwargs):
    rewards = []
    for completion, tests in zip(completions, test_list):
        text = completion[0]["content"]
        code = extract_code(text)
        if not code:
            rewards.append(0.0)
            continue
        passed = execute_code_with_tests(code, tests)
        rewards.append(1.0 if passed else 0.0)
    return rewards

def format_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        has_code_block = bool(re.search(r"```(?:python)?\s*\n.+?```", text, re.DOTALL))
        rewards.append(0.5 if has_code_block else 0.0)
    return rewards

def syntax_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion[0]["content"]
        code = extract_code(text)
        if not code:
            rewards.append(0.0)
            continue
        try:
            compile(code, "<string>", "exec")
            rewards.append(0.25)
        except SyntaxError:
            rewards.append(0.0)
    return rewards

# Simulate a batch of completions (mix of correct, wrong, gibberish)
ex = train_data[0]
fake_completions = [
    # Correct: ground-truth code in a fenced block
    [{"content": f"```python\n{ex['code']}\n```"}],
    # Wrong code in a fenced block
    [{"content": "```python\ndef wrong():\n    return 42\n```"}],
    # Syntax error
    [{"content": "```python\ndef broken(\n```"}],
    # No code at all
    [{"content": "I have no idea how to solve this."}],
]
fake_tests = [ex["test_list"]] * 4

print("=== Reward function test ===")
print(f"Task: {ex['text'][:100]}...")
print(f"Tests: {ex['test_list']}")
print()

cr = correctness_reward(fake_completions, fake_tests)
fr = format_reward(fake_completions)
sr = syntax_reward(fake_completions)

labels = ["correct+fenced", "wrong+fenced", "syntax_err+fenced", "no_code"]
print(f"{'Completion':<22} {'Correct':>8} {'Format':>8} {'Syntax':>8} {'Total':>8}")
print("-" * 60)
for label, c, f, s in zip(labels, cr, fr, sr):
    print(f"{label:<22} {c:>8.2f} {f:>8.2f} {s:>8.2f} {c+f+s:>8.2f}")

In [ ]:
# Cell 5: Format dataset for GRPOTrainer and inspect prompts
SYSTEM_PROMPT = (
    "You are an expert Python programmer. "
    "Write a Python function to solve the given task. "
    "Put your code in a ```python code block."
)

def format_example(example):
    example["prompt"] = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["text"]},
    ]
    return example

train_dataset = train_data.map(format_example)
test_dataset = test_data.map(format_example)

print(f"Train: {len(train_dataset)}, Test: {len(test_dataset)}")
print(f"Columns: {train_dataset.column_names}")
print(f"\n--- Sample prompt ---")
print(train_dataset[0]["prompt"])

In [ ]:
# Cell 6: Quick model inference test — generate a completion and score it
# (runs on CPU/MPS locally, or GPU if available)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype).to(device)
model.eval()
print(f"Model loaded on {device}")

In [ ]:
# Cell 7: Generate completions for a few MBPP tasks and score them
import time

results = []
for i in range(5):
    ex = train_dataset[i]
    prompt = tokenizer.apply_chat_template(
        ex["prompt"], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    t0 = time.time()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
        )
    elapsed = time.time() - t0
    
    completion_ids = output[0][inputs["input_ids"].shape[1]:]
    completion_text = tokenizer.decode(completion_ids, skip_special_tokens=True)
    
    # Score it
    code = extract_code(completion_text)
    passed = execute_code_with_tests(code, ex["test_list"]) if code else False
    has_block = bool(re.search(r"```(?:python)?\s*\n.+?```", completion_text, re.DOTALL))
    try:
        compile(code, "<string>", "exec")
        valid_syntax = True
    except SyntaxError:
        valid_syntax = False
    
    results.append({
        "task_id": ex["task_id"],
        "task": ex["text"][:80],
        "correct": passed,
        "has_block": has_block,
        "valid_syntax": valid_syntax,
        "time": elapsed,
    })
    
    status = "PASS" if passed else "FAIL"
    print(f"\n{'='*60}")
    print(f"[task_id={ex['task_id']}] {status} | {ex['text'][:80]}...")
    print(f"  format: {'```python```' if has_block else 'raw'} | syntax: {'ok' if valid_syntax else 'err'} | {elapsed:.1f}s")
    print(f"  tests: {ex['test_list'][:2]}")
    print(f"  --- completion (first 300 chars) ---")
    print(f"  {completion_text[:300]}")

# Summary
n_correct = sum(r["correct"] for r in results)
n_syntax = sum(r["valid_syntax"] for r in results)
n_format = sum(r["has_block"] for r in results)
print(f"\n{'='*60}")
print(f"Summary: {n_correct}/{len(results)} correct, {n_syntax}/{len(results)} valid syntax, {n_format}/{len(results)} fenced block")

In [ ]:
# Cell 8: Verify reward functions work with GRPOTrainer-style batch format
# This simulates what GRPOTrainer passes to reward_funcs

print("=== Simulating GRPOTrainer reward call format ===\n")

# GRPOTrainer passes: completions as list of [{"role":"assistant","content":...}]
# plus dataset columns as kwargs (here: test_list, text, task_id, code, prompt)
batch_completions = []
batch_test_list = []

for i in range(4):
    ex = train_dataset[i]
    # Use ground-truth code as "completion" for 2, garbage for 2
    if i < 2:
        content = f"```python\n{ex['code']}\n```"
    else:
        content = "I don't know"
    batch_completions.append([{"role": "assistant", "content": content}])
    batch_test_list.append(ex["test_list"])

cr = correctness_reward(batch_completions, test_list=batch_test_list)
fr = format_reward(batch_completions)
sr = syntax_reward(batch_completions)

print(f"correctness_reward: {cr}")
print(f"format_reward:      {fr}")
print(f"syntax_reward:      {sr}")
print(f"\nExpected: first 2 should score high, last 2 should score ~0")

# GRPO Code Task — Test Notebook\n\nVerify the MBPP dataset loading, code extraction, execution-based reward, and a quick training smoke test.